# Package 5 – Follow-up Policy Analysis: "Are Late Follow-ups a Waste of Time?"

The sales manager believes there is no value in following up after the 3rd call. This notebook checks that claim against the data.

**No recomputation here** — every table, chart, and statistic below is imported directly from `follow_up_analysis.py` (importing it re-runs its cells once, which is how the results become available). This notebook only adds richer presentation and business narrative.

**Limitation stated up front:** the dataset has no column linking a specific closed deal to a specific follow-up stage. So this analysis relies on two data-supported angles only: the stage-by-stage lead-retention/dropout curve, and `calls_to_closed` vs. `calls_to_not_closed` as an aggregate proxy for contact effort. It does not — and cannot — claim "N deals closed after the 4th call."

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

from follow_up_analysis import (
    call_stats,
    dropout_stats,
    late_stage_remaining,
    total_closed,
)

sns.set_theme(style="whitegrid")

Shape: (3490, 19)
Total closed deals: 10557
Follow-up 3: 46323 leads remaining -> closed deals are 22.8% of that pool
Follow-up 4: 41517 leads remaining -> closed deals are 25.4% of that pool
Follow-up 5: 29384 leads remaining -> closed deals are 35.9% of that pool
Average calls to close a deal:      3.52
Average calls before giving up:     3.93
Difference (closed - not closed):   -0.41
CONCLUSION:
Follow-up 4 and Follow-up 5 still show a 10.4% and 29.2% drop-off respectively — leads are still being lost (and by extension, still being actively worked) well past the 3rd call. Separately, closed deals require 3.52 calls on average vs. 3.93 for deals that were abandoned — closed deals take FEWER calls on average, meaning deals that need many calls are less likely to close, supporting the case for cutting off effort earlier. Note: the data cannot attribute a specific close to a specific follow-up stage, so this conclusion rests on the retention curve and the calls-to-outcome comparison, no

C:\Users\lizar\OneDrive\Desktop\AI DEVELOPER\funnel_marketing_data\follow_up_analysis.py:121: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 1–2: Follow-up Funnel & Dropout Rate

In [2]:
dropout_stats.style.background_gradient(
    subset=["Drop Rate (%)"], cmap="Reds"
).format(precision=1, na_rep="—")

,Stage,Remaining Leads,Drop From Previous Stage,Drop Rate (%)
0,Leads Answered,97843,—,—
1,Follow-up 1,76575,21268.0,21.7
2,Follow-up 2,56916,19659.0,25.7
3,Follow-up 3,46323,10593.0,18.6
4,Follow-up 4,41517,4806.0,10.4
5,Follow-up 5,29384,12133.0,29.2


Drop-off is **not** a smooth, steadily-declining curve. It falls from Follow-up 1 (21.7%) to Follow-up 2 (25.7%, actually the highest of the early stages) to Follow-up 3 (18.6%), dips to its **lowest point at Follow-up 4 (10.4%)** — then **spikes to its highest point of any stage at Follow-up 5 (29.2%)**. That Follow-up 5 spike is the "unexpected" stage: after three rounds of comparatively steady attrition, the sales team apparently pushes hard through a 4th call (low drop-off) and then loses nearly a third of what's left on the 5th.

## Step 3: Visualizations

In [3]:
plt.figure(figsize=(8, 6))
sns.lineplot(data=dropout_stats, x="Stage", y="Remaining Leads", marker="o")
plt.title("Lead Retention Across Follow-up Stages")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

C:\Users\lizar\AppData\Local\Temp\ipykernel_27988\1266396778.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
drop_only = dropout_stats.dropna(subset=["Drop Rate (%)"])

plt.figure(figsize=(8, 6))
sns.barplot(data=drop_only, x="Stage", y="Drop Rate (%)")
plt.title("Lead Drop-off Rate by Follow-up Stage")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

C:\Users\lizar\AppData\Local\Temp\ipykernel_27988\3628320226.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# No native funnel-chart type in matplotlib/seaborn — a horizontal bar chart
# ordered by stage is the standard fallback for this shape.
plt.figure(figsize=(8, 6))
funnel_order = dropout_stats.iloc[::-1]
sns.barplot(data=funnel_order, y="Stage", x="Remaining Leads", orient="h")
plt.title("Follow-up Funnel: Leads Remaining by Stage")
plt.tight_layout()
plt.show()

C:\Users\lizar\AppData\Local\Temp\ipykernel_27988\3554079153.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 4: Late Follow-up Analysis

In [6]:
print(f"Total closed deals (dataset-wide): {total_closed}")
late_stage_remaining

Total closed deals (dataset-wide): 10557


Stage
Follow-up 3    46323
Follow-up 4    41517
Follow-up 5    29384
Name: Remaining Leads, dtype: int64

At Follow-up 3, 4, and 5 there are still 46,323 / 41,517 / 29,384 leads remaining respectively — far more than the 10,557 total closed deals. In other words, the pool of leads still "in play" after the 3rd call remains large in absolute terms all the way through Follow-up 5, so there's no sign of the funnel having already exhausted its useful leads by round 3.

**Caution on interpretation:** the "closed deals as % of remaining pool" figures printed by `follow_up_analysis.py` (22.8% → 25.4% → 35.9% across stages 3–5) rise mechanically as the remaining-lead denominator shrinks, since `closed` is a single dataset-wide total, not a stage-specific count. That rising percentage is **not** evidence that later follow-ups convert better — it's an artifact of a shrinking pool relative to a fixed numerator. It should be read only as "the remaining pool is still large enough to plausibly contain the closed deals," not as a stage-level conversion rate.

## Step 5: Sales Call Analysis

In [7]:
plt.figure(figsize=(6, 6))
sns.barplot(
    x=["Calls to Closed", "Calls to Not Closed"],
    y=[call_stats["avg_calls_to_closed"], call_stats["avg_calls_to_not_closed"]],
)
plt.ylabel("Average number of calls")
plt.title("Average Calls: Closed vs. Not Closed Deals")
plt.tight_layout()
plt.show()

print(f"Average calls to close a deal:    {call_stats['avg_calls_to_closed']:.2f}")
print(f"Average calls before giving up:   {call_stats['avg_calls_to_not_closed']:.2f}")
print(f"Difference (closed - not closed): {call_stats['difference']:+.2f}")

Average calls to close a deal:    3.52
Average calls before giving up:   3.93
Difference (closed - not closed): -0.41


C:\Users\lizar\AppData\Local\Temp\ipykernel_27988\2292780497.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Do successful customers need more or fewer calls?** Fewer — closed deals take **3.52 calls on average**, versus **3.93 calls** for deals that were eventually abandoned. The gap is real but modest (≈0.41 calls): deals that drag on longer are somewhat less likely to close than deals that resolve relatively quickly, but this is a mild tendency, not a sharp cliff at any particular call count.

## Step 6: Business Conclusion

**Is the sales manager right that follow-ups after the 3rd call are a waste of time?** Mostly not, but with a real nuance worth acting on:

- The retention curve shows the funnel is still clearly "active" well past the 3rd call — Follow-up 4 has the *lowest* drop-off of any stage (10.4%), meaning leads are still being successfully retained and worked at that point. That directly contradicts a blanket "stop after 3" rule.
- However, Follow-up 5 is the **worst-performing stage in the entire funnel** (29.2% drop-off, the highest of all five stages) — so if there's a point to reconsider policy, it's not "after the 3rd call," it's specifically **the 5th call**, where the data shows a real and unusual spike in loss.
- The calls-to-outcome comparison adds a second, independent signal in the same direction: deals that ultimately close need modestly *fewer* calls (3.52) than deals that get abandoned (3.93) — so pushing a deal into a high call count is a mild, but not decisive, warning sign.

**Bottom line:** the data does not support cutting off effort after the 3rd call — Follow-up 4 is actually the strongest-retaining stage in the funnel. The sales manager's instinct is better aimed at the 5th call specifically, where both an unusual drop-off spike and the calls-to-outcome data point in the same direction.

## Step 7: Recommendations for Northbound Media

- **Do not stop after the 3rd call.** Follow-up 4 has the lowest drop-off of any stage in the funnel — cutting the sequence there would forgo the stage where the sales team is currently most successful at retaining leads.
- **Investigate the Follow-up 5 spike specifically.** A 29.2% drop-off — nearly 3x the Follow-up 4 rate — is the one genuine anomaly in this funnel. Before changing broad policy, Northbound should look at *why* the 5th call underperforms: is it a scripting/training issue, lead fatigue, or a timing problem (e.g. calls spaced too far apart by that point)?
- **Use call count as a soft signal, not a hard cutoff.** Since closed deals average 3.52 calls vs. 3.93 for abandoned ones, a lead that's still open past ~4 calls is statistically less likely to close — worth flagging for a manager review or a different outreach approach, rather than an automatic drop.
- **Target the policy change at the mechanism, not the count.** Because the data can't attribute closes to a specific stage, the responsible next step is a controlled test (e.g. A/B a revised 5th-call script or timing) rather than removing the stage outright.

## Final Report Summary

*(Copy-ready for `README.md`.)*

```markdown
# Package 5 – Follow-up Analysis

## Objective

Northbound's sales manager claims follow-up calls after the 3rd call are a
waste of time. This analysis checks that claim against the funnel data —
the dataset has no column linking a specific closed deal to a specific
follow-up stage, so the check relies on the stage-by-stage lead-retention
curve and the calls-to-closed vs. calls-to-not-closed comparison, not on an
invented per-stage close count.

## Methodology

Dropout rates were calculated as `(previous stage - current stage) /
previous stage`, using `leads_answered` as the stage-0 baseline and summing
`followup_1`–`followup_5` across the dataset (each row is a cohort-level
record, consistent with how earlier packages treat these columns). Call
effort was compared via the mean of `calls_to_closed` vs.
`calls_to_not_closed`.

## Key Findings

- Drop-off is not smoothly declining: Follow-up 4 has the **lowest**
  drop-off of any stage (10.4%), while Follow-up 5 has the **highest**
  (29.2%) — a clear anomaly worth investigating on its own.
- Leads remaining at Follow-up 3/4/5 (46,323 / 41,517 / 29,384) stay far
  larger than the total closed-deal count (10,557) throughout, so the
  funnel is still actively working well past the 3rd call.
- Closed deals average fewer calls (3.52) than abandoned deals (3.93) — a
  real but modest signal that dragging a deal out is mildly associated
  with a lower chance of closing.

## Business Insight

Additional follow-ups do create measurable value past the 3rd call —
Follow-up 4 is the strongest-retaining stage in the entire funnel. The one
genuine weak point is Follow-up 5 specifically, not "anything after call 3."

## Recommendation

Keep the full 5-call sequence; do not cut off after the 3rd call. Instead,
investigate and address the Follow-up 5 drop-off spike directly (script,
timing, or lead-fatigue review), and use rising call count past ~4 calls as
a soft prioritization signal rather than a hard policy cutoff.
```